In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;


# Module 2: 데이터 탐색 (EDA)

TPC-H 데이터를 탐색하고, ML 모델링에 적합한 피처 후보를 파악합니다.

### 학습 목표
- Snowpark DataFrame API로 대규모 데이터를 Snowflake 내에서 탐색
- TPC-H 테이블 구조 이해 (CUSTOMER, ORDERS, LINEITEM)
- 피처 후보 및 타겟 변수 설계

### 데이터 소스

| 테이블 | 위치 | 행 수 |
|--------|------|-------|
| CUSTOMER | SNOWFLAKE_SAMPLE_DATA.TPCH_SF1 | 150,000 |
| ORDERS | SNOWFLAKE_SAMPLE_DATA.TPCH_SF1 | 1,500,000 |
| LINEITEM | SNOWFLAKE_SAMPLE_DATA.TPCH_SF1 | 6,000,000 |

### 시간 구조

```
1992 ─────────────── 1997.06.30 ─────────── 1998.06.30
│      관찰기간        │     예측 윈도우      │
│   (피처 계산 구간)    │  (타겟: 12개월 구매) │
```


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()
print('Session ready')


## 1. 데이터 로드


In [ ]:
df_customer = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER')
df_orders = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS')
df_lineitem = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM')

print(f'CUSTOMER:  {df_customer.count():>10,} rows')
print(f'ORDERS:    {df_orders.count():>10,} rows')
print(f'LINEITEM:  {df_lineitem.count():>10,} rows')


## 2. 스키마 탐색


In [ ]:
print('=== CUSTOMER ===')
df_customer.printSchema()
df_customer.show(5)

print('\n=== ORDERS ===')
df_orders.printSchema()
df_orders.show(5)

print('\n=== LINEITEM (first 5 cols) ===')
df_lineitem.select('L_ORDERKEY','L_PARTKEY','L_QUANTITY','L_EXTENDEDPRICE','L_DISCOUNT').show(5)


## 3. 기술 통계


In [ ]:
# ORDERS 날짜 범위
date_stats = df_orders.agg(
    F.min('O_ORDERDATE').alias('MIN_DATE'),
    F.max('O_ORDERDATE').alias('MAX_DATE'),
    F.count('*').alias('TOTAL')
).to_pandas()
print('ORDERS date range:')
print(date_stats.to_string(index=False))

# CUSTOMER 기술 통계
print('\nCUSTOMER C_ACCTBAL stats:')
df_customer.select('C_ACCTBAL').describe().show()


## 4. 시각화


In [ ]:
# Market Segment Distribution
seg_pd = df_customer.group_by('C_MKTSEGMENT').count().to_pandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(seg_pd['C_MKTSEGMENT'], seg_pd['COUNT'], color='#636EFA', edgecolor='white')
ax.set_title('Customer Market Segment Distribution')
ax.set_xlabel('Segment')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Order Amount Distribution (5% sample)
orders_sample = df_orders.sample(0.05).select('O_TOTALPRICE').to_pandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(orders_sample['O_TOTALPRICE'], bins=50, color='#00CC96', edgecolor='white')
ax.set_title('Order Amount Distribution (5% Sample)')
ax.set_xlabel('Total Price ($)')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()


## 5. 피처 설계

다음 모듈(Feature Store)에서 생성할 피처 목록:

| 피처명 | 소스 | 설명 |
|--------|------|------|
| C_MKTSEGMENT | CUSTOMER | 시장 세그먼트 (범주형) |
| C_ACCTBAL | CUSTOMER | 계정 잔액 |
| C_NATIONKEY | CUSTOMER | 국가 키 |
| TOTAL_ORDERS | ORDERS | 누적 주문 횟수 |
| AVG_ORDER_VALUE | ORDERS | 평균 주문 금액 |
| TOTAL_REVENUE | ORDERS+LINEITEM | 총 매출액 |
| AVG_DISCOUNT | LINEITEM | 평균 할인율 |
| AVG_QUANTITY | LINEITEM | 평균 주문 수량 |
| DAYS_SINCE_LAST_ORDER | ORDERS | 마지막 주문 후 경과일 |

**타겟**: `FUTURE_12M_REVENUE` — 예측 윈도우(1997.07~1998.06) 12개월 구매금액

### 다음 단계

Module 3에서 이 피처들을 **일별 스냅샷 테이블**로 생성하고 Feature Store에 등록합니다.
